# AdventureWorks Sales Analysis

SQL Server + Pandas + SciPy + Matplotlib. Uses `Sales.SalesOrderHeader` and calculates a 95% t confidence interval for mean **SubTotal** (excludes tax and freight). Run cells top-to-bottom with access to AdventureWorks2022.

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

## Connection
Set `SQL_SERVER` in your environment. Windows integrated authentication is used, matching the original project. Requires SQL Server ODBC Driver and `pyodbc`. No local machine name is hardcoded.

In [ ]:
server = os.environ.get("SQL_SERVER")
if not server:
    raise RuntimeError("Set SQL_SERVER to your accessible SQL Server host before running.")
url = URL.create("mssql+pyodbc", host=server, database="AdventureWorks2022", query={"driver": os.environ.get("SQL_ODBC_DRIVER", "ODBC Driver 17 for SQL Server"), "trusted_connection": "yes"})
engine = create_engine(url)

## Extract and validate orders

In [ ]:
query = text("""SELECT SalesOrderID, OrderDate, CustomerID, TerritoryID, SubTotal, TotalDue FROM Sales.SalesOrderHeader""")
with engine.connect() as connection:
    orders = pd.read_sql(query, connection)
orders["OrderDate"] = pd.to_datetime(orders["OrderDate"])
orders["SubTotal"] = pd.to_numeric(orders["SubTotal"], errors="coerce")
assert orders["SalesOrderID"].is_unique, "Duplicate order IDs"
assert orders["SubTotal"].notna().all(), "Missing order subtotals"
assert (orders["SubTotal"] >= 0).all(), "Negative order subtotals"
print(f"Orders: {len(orders):,}; date range: {orders.OrderDate.min().date()} to {orders.OrderDate.max().date()}")

## Monthly KPIs

In [ ]:
orders["Month"] = orders["OrderDate"].dt.to_period("M").dt.to_timestamp()
monthly = orders.groupby("Month").agg(Orders=("SalesOrderID", "count"), Revenue=("SubTotal", "sum"), AverageOrderValue=("SubTotal", "mean")).reset_index()
display(monthly.tail(12))
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(monthly["Month"], monthly["Revenue"])
ax.set(title="Monthly sales revenue (SubTotal)", xlabel="Month", ylabel="Revenue")
fig.autofmt_xdate()
plt.show()

## Mean order subtotal with 95% confidence interval
This interval quantifies uncertainty in the estimated population mean **under the assumptions** of the t interval; it does not demonstrate the effectiveness of any business strategy. Orders may be dependent, and this dataset may represent a complete historical population rather than a sample.

In [ ]:
values = orders["SubTotal"].dropna().to_numpy(dtype=float)
n = values.size
if n < 2:
    raise ValueError("At least two observations required")
mean = np.mean(values)
sem = stats.sem(values)
lo, hi = stats.t.interval(0.95, df=n - 1, loc=mean, scale=sem)
print(f"n={n:,}, mean SubTotal={mean:,.2f}; 95% t-CI [{lo:,.2f}, {hi:,.2f}]")

## Limitations
No database included in this repository. Results require execution against the relevant AdventureWorks2022 version. Revenue is reported using `SubTotal`; `TotalDue` includes tax/freight. Full- and partial-year comparisons must be separated.